In [1]:
# Step 1: Import the tools we need
import pandas as pd
import sqlite3

# Step 2: Load the data
# We are reading the csv file you uploaded
df = pd.read_csv('supply_chain_data.csv')

# Step 3: Calculate "Run Rate" (Daily Sales)
# We assume 'Number of products sold' happens over a typical month (30 days)
# Run Rate = How many we sell per day on average
df['Run_Rate_Daily'] = df['Number of products sold'] / 30

# Step 4: Calculate "Months of Inventory" (How long stock will last)
# If we have 100 items and sell 5 a day, stock lasts 20 days.
# We add a small number (0.01) to avoid dividing by zero error.
df['Months_Inventory_On_Hand'] = df['Stock levels'] / df['Number of products sold']

# Show the first 5 rows to check our work
print("Data Loaded Successfully! Here are the first 5 rows with new metrics:")
display(df[['Product type', 'SKU', 'Stock levels', 'Number of products sold', 'Run_Rate_Daily', 'Months_Inventory_On_Hand']].head())

Data Loaded Successfully! Here are the first 5 rows with new metrics:


,Product type,SKU,Stock levels,Number of products sold,Run_Rate_Daily,Months_Inventory_On_Hand
0,haircare,SKU0,58,802,26.733333,0.072319
1,skincare,SKU1,53,736,24.533333,0.072011
2,haircare,SKU2,1,8,0.266667,0.125000
3,skincare,SKU3,23,83,2.766667,0.277108
4,skincare,SKU4,5,871,29.033333,0.005741


In [2]:
import sqlite3

# 1. Create a virtual SQL database in memory (RAM)
conn = sqlite3.connect(':memory:')

# 2. Push your Python data (df) into this SQL database as a table named 'inventory_table'
df.to_sql('inventory_table', conn, index=False, if_exists='replace')

# 3. The "Money" Query: Categorize Inventory using SQL Logic
# We use a CASE statement to label products automatically.
sql_query = """
SELECT
    "Product type",
    SKU,
    Price,
    "Stock levels",
    "Number of products sold",
    Run_Rate_Daily,
    Months_Inventory_On_Hand,
    CASE
        -- Logic: If stock is high (>80) but sales are low (<200), it's "Dead Stock"
        WHEN "Stock levels" > 80 AND "Number of products sold" < 200 THEN 'Overstocked (Cash Trap)'

        -- Logic: If stock is low (<30) but sales are high (>500), we might run out!
        WHEN "Stock levels" < 30 AND "Number of products sold" > 500 THEN 'Stock Out Risk (Hot Item)'

        -- Everything else is considered normal
        ELSE 'Healthy'
    END AS Inventory_Status
FROM inventory_table
ORDER BY "Number of products sold" DESC
"""

# 4. Run the query and save it back to a variable
final_data = pd.read_sql(sql_query, conn)

# 5. Show the "Advanced" results
print("SQL Query Executed Successfully! Here is your categorized data:")
display(final_data.head(10))

SQL Query Executed Successfully! Here is your categorized data:


,Product type,SKU,Price,Stock levels,Number of products sold,Run_Rate_Daily,Months_Inventory_On_Hand,Inventory_Status
0,skincare,SKU10,15.707796,51,996,33.200000,0.051205,Healthy
1,cosmetics,SKU94,3.037689,77,987,32.900000,0.078014,Healthy
2,skincare,SKU9,64.015733,14,980,32.666667,0.014286,Stock Out Risk (Hot Item)
3,skincare,SKU36,9.813003,18,963,32.100000,0.018692,Stock Out Risk (Hot Item)
4,skincare,SKU37,23.399845,25,963,32.100000,0.025961,Stock Out Risk (Hot Item)
5,skincare,SKU11,90.635460,46,960,32.000000,0.047917,Healthy
6,haircare,SKU78,6.306883,5,946,31.533333,0.005285,Stock Out Risk (Hot Item)
7,skincare,SKU40,80.541424,90,933,31.100000,0.096463,Healthy
8,cosmetics,SKU44,51.355791,13,919,30.633333,0.014146,Stock Out Risk (Hot Item)
9,cosmetics,SKU91,62.111965,98,916,30.533333,0.106987,Healthy


In [3]:
from google.colab import files

# Save our engineered data to a new CSV
final_data.to_csv('advanced_supply_chain_output.csv', index=False)

# Download it to your computer automatically
files.download('advanced_supply_chain_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import pandas as pd
import numpy as np
import io

# 1. Load the Data
df = pd.read_csv('supply_chain_data.csv')

# --- FEATURE 1: ABC ANALYSIS (The "80/20" Rule) ---
# Calculate Revenue
df['Revenue'] = df['Price'] * df['Number of products sold']

# Sort by Revenue (Highest to Lowest)
df = df.sort_values(by='Revenue', ascending=False)

# Calculate Cumulative Revenue Share
df['Cumulative_Revenue'] = df['Revenue'].cumsum()
df['Total_Revenue'] = df['Revenue'].sum()
df['Revenue_Share_Pct'] = df['Cumulative_Revenue'] / df['Total_Revenue']

# Assign A, B, C Classes
# A = Top 80% of money, B = Next 15%, C = Bottom 5%
def get_abc_class(percentage):
    if percentage <= 0.80:
        return 'A (High Value)'
    elif percentage <= 0.95:
        return 'B (Medium Value)'
    else:
        return 'C (Low Value)'

df['ABC_Class'] = df['Revenue_Share_Pct'].apply(get_abc_class)

# --- FEATURE 2: PREPARE FOR FORECASTING (Synthetic Dates) ---
# The original data has no dates, so we simulate "Date of Sale" over the last year
# to enable Power BI Forecasting.
start_date = pd.to_datetime('2023-01-01')
# Generate random days added to start date
random_days = np.random.randint(0, 365, size=len(df))
df['Date_of_Sale'] = start_date + pd.to_timedelta(random_days, unit='D')

# --- FEATURE 3: RISK FLAGS (Your Logic) ---
# Re-calculating your custom metrics
df['Run_Rate_Daily'] = df['Number of products sold'] / 30
df['Days_Inventory_On_Hand'] = (df['Stock levels'] / df['Number of products sold']) * 30

def get_risk_status(row):
    if row['Stock levels'] > 80 and row['Number of products sold'] < 200:
        return 'Overstocked (Cash Trap)'
    elif row['Stock levels'] < 30 and row['Number of products sold'] > 500:
        return 'Stock Out Risk (Hot Item)'
    else:
        return 'Healthy'

df['Inventory_Status'] = df.apply(get_risk_status, axis=1)

# --- EXPORT ---
# Save the "Super-Advanced" file
df.to_csv('final_supply_chain_project.csv', index=False)
print("Success! Download 'final_supply_chain_project.csv' from the files menu.")

Success! Download 'final_supply_chain_project.csv' from the files menu.
